In [1]:
import os
import json
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("✓ Imports complete")

C:\Users\Mansoor\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\Mansoor\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


✓ Imports complete


In [3]:
df_train = pd.read_parquet("../data/processed/splits/train.parquet")
df_val   = pd.read_parquet("../data/processed/splits/val.parquet")
df_test  = pd.read_parquet("../data/processed/splits/test.parquet")

# Restore timestamps
ts_cols = [
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for df in [df_train, df_val, df_test]:
    for col in ts_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

print("✓ All splits loaded and timestamps restored")
print(f"  Train : {df_train.shape}")
print(f"  Val   : {df_val.shape}")
print(f"  Test  : {df_test.shape}")

✓ All splits loaded and timestamps restored
  Train : (67526, 40)
  Val   : (14473, 40)
  Test  : (14471, 40)


In [4]:
# From EDA: 29 orders > +100 days, 3 orders < -100 days
# These are almost certainly data entry errors
# Only drop from train — val/test kept as-is for honest evaluation

before = len(df_train)
df_train = df_train[
    (df_train["lead_time_variance"] >= -100) &
    (df_train["lead_time_variance"] <= 100)
].copy()
after = len(df_train)

print("STEP 1 — Drop extreme target outliers (train only)")
print(f"  Threshold        : [-100, +100] days")
print(f"  Train rows before: {before:,}")
print(f"  Train rows after : {after:,}")
print(f"  Rows dropped     : {before - after}")
print(f"\n  New target stats (train):")
print(df_train["lead_time_variance"].describe().round(4).to_string())

STEP 1 — Drop extreme target outliers (train only)
  Threshold        : [-100, +100] days
  Train rows before: 67,526
  Train rows after : 67,494
  Rows dropped     : 32

  New target stats (train):
count    67494.0000
mean       -11.2353
std          9.6581
min        -83.0706
25%        -16.2448
50%        -11.9889
75%         -6.4008
max         97.9718


In [5]:
# From EDA: 1 row with null payment columns — drop from all splits
print("STEP 2 — Drop null payment rows")

for name, df in [("train", df_train), ("val", df_val), ("test", df_test)]:
    before = len(df)
    mask = df["total_payment_value"].isnull()
    if name == "train":
        df_train = df_train[~mask].copy()
        print(f"  Train: dropped {mask.sum()} rows → {len(df_train):,} remain")
    elif name == "val":
        df_val = df_val[~mask].copy()
        print(f"  Val  : dropped {mask.sum()} rows → {len(df_val):,} remain")
    else:
        df_test = df_test[~mask].copy()
        print(f"  Test : dropped {mask.sum()} rows → {len(df_test):,} remain")

STEP 2 — Drop null payment rows
  Train: dropped 1 rows → 67,493 remain
  Val  : dropped 0 rows → 14,473 remain
  Test : dropped 0 rows → 14,471 remain


In [6]:
# Strategy: fill with "unknown" — ~1.4% of rows
# Fit decision made from train EDA, applied identically to all splits

CATEGORY_FILL = "unknown"

print("STEP 3 — Impute null product categories")
for split_name, df in [("train", df_train), ("val", df_val), ("test", df_test)]:
    before = df["product_category_name_english"].isnull().sum()
    df["product_category_name_english"] = df["product_category_name_english"].fillna(CATEGORY_FILL)
    df["product_category_name"] = df["product_category_name"].fillna(CATEGORY_FILL)
    after = df["product_category_name_english"].isnull().sum()
    print(f"  {split_name:<6}: {before} nulls → {after} nulls (filled with '{CATEGORY_FILL}')")

STEP 3 — Impute null product categories
  train : 940 nulls → 0 nulls (filled with 'unknown')
  val   : 217 nulls → 0 nulls (filled with 'unknown')
  test  : 220 nulls → 0 nulls (filled with 'unknown')


In [7]:
# Strategy: median imputation per category — fit medians on train only
# 12 rows in train with null dimensions

dim_cols = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]

print("STEP 4 — Impute product dimension nulls (median per category, fit on train)\n")

# Fit: compute median per category on train
dim_medians = (
    df_train.groupby("product_category_name_english")[dim_cols]
    .median()
    .to_dict()
)

# Fallback: global median from train (for categories not in train)
global_medians = {col: df_train[col].median() for col in dim_cols}

def impute_dimensions(df, dim_medians, global_medians, dim_cols):
    df = df.copy()
    for col in dim_cols:
        null_mask = df[col].isnull()
        if null_mask.sum() == 0:
            continue
        # Fill using category median
        cat_fill = df.loc[null_mask, "product_category_name_english"].map(
            dim_medians[col]
        )
        df.loc[null_mask, col] = cat_fill
        # Any still null → global median
        still_null = df[col].isnull()
        df.loc[still_null, col] = global_medians[col]
    return df

df_train = impute_dimensions(df_train, dim_medians, global_medians, dim_cols)
df_val   = impute_dimensions(df_val,   dim_medians, global_medians, dim_cols)
df_test  = impute_dimensions(df_test,  dim_medians, global_medians, dim_cols)

for split_name, df in [("train", df_train), ("val", df_val), ("test", df_test)]:
    nulls = df[dim_cols].isnull().sum().sum()
    print(f"  {split_name:<6}: remaining nulls in dim cols = {nulls}")

print(f"\n  Global fallback medians (from train):")
for col, val in global_medians.items():
    print(f"    {col:<25} : {val:.2f}")

STEP 4 — Impute product dimension nulls (median per category, fit on train)

  train : remaining nulls in dim cols = 0
  val   : remaining nulls in dim cols = 0
  test  : remaining nulls in dim cols = 0

  Global fallback medians (from train):
    product_weight_g          : 700.00
    product_length_cm         : 25.00
    product_height_cm         : 13.00
    product_width_cm          : 20.00


In [8]:
# product_name_lenght, product_description_lenght, product_photos_qty
# Strategy: median per category — fit on train only

text_cols = ["product_name_lenght", "product_description_lenght", "product_photos_qty"]

print("STEP 5 — Impute product text feature nulls (median per category, fit on train)\n")

text_medians = (
    df_train.groupby("product_category_name_english")[text_cols]
    .median()
    .to_dict()
)
global_text_medians = {col: df_train[col].median() for col in text_cols}

def impute_text_features(df, text_medians, global_text_medians, text_cols):
    df = df.copy()
    for col in text_cols:
        null_mask = df[col].isnull()
        if null_mask.sum() == 0:
            continue
        cat_fill = df.loc[null_mask, "product_category_name_english"].map(
            text_medians[col]
        )
        df.loc[null_mask, col] = cat_fill
        still_null = df[col].isnull()
        df.loc[still_null, col] = global_text_medians[col]
    return df

df_train = impute_text_features(df_train, text_medians, global_text_medians, text_cols)
df_val   = impute_text_features(df_val,   text_medians, global_text_medians, text_cols)
df_test  = impute_text_features(df_test,  text_medians, global_text_medians, text_cols)

for split_name, df in [("train", df_train), ("val", df_val), ("test", df_test)]:
    nulls = df[text_cols].isnull().sum().sum()
    print(f"  {split_name:<6}: remaining nulls in text cols = {nulls}")

print(f"\n  Global fallback medians (from train):")
for col, val in global_text_medians.items():
    print(f"    {col:<35} : {val:.2f}")

STEP 5 — Impute product text feature nulls (median per category, fit on train)

  train : remaining nulls in text cols = 0
  val   : remaining nulls in text cols = 0
  test  : remaining nulls in text cols = 0

  Global fallback medians (from train):
    product_name_lenght                 : 52.00
    product_description_lenght          : 608.00
    product_photos_qty                  : 2.00


In [9]:
# review_score, review_count, has_review_comment are POST-DELIVERY
# They will NOT be used as features — but we clean them anyway
# to keep the dataframe consistent
# Strategy: fill review_score with train median, others with 0

print("STEP 6 — Handle review nulls (post-delivery cols — excluded from features)\n")

review_score_median = df_train["review_score"].median()

for split_name, df in [("train", df_train), ("val", df_val), ("test", df_test)]:
    before = df["review_score"].isnull().sum()
    df["review_score"]       = df["review_score"].fillna(review_score_median)
    df["review_count"]       = df["review_count"].fillna(0)
    df["has_review_comment"] = df["has_review_comment"].fillna(0)
    after = df["review_score"].isnull().sum()
    print(f"  {split_name:<6}: review_score nulls {before} → {after}")

print(f"\n  review_score fill value (train median): {review_score_median}")
print(f"  Note: These columns will NOT be used as model features.")

STEP 6 — Handle review nulls (post-delivery cols — excluded from features)

  train : review_score nulls 449 → 0
  val   : review_score nulls 99 → 0
  test  : review_score nulls 96 → 0

  review_score fill value (train median): 5.0
  Note: These columns will NOT be used as model features.


In [10]:
# Strategy: cap at [1st, 99th] percentile — fit caps on train only
# Columns to cap based on EDA outlier analysis

cap_cols = [
    "total_price", "total_freight", "avg_price",
    "product_weight_g", "product_length_cm",
    "product_height_cm", "product_width_cm",
    "total_payment_value", "product_description_lenght",
    "product_photos_qty", "product_name_lenght",
]

print("STEP 7 — Cap outliers at [1st, 99th] percentile (fit on train)\n")
print(f"  {'Column':<35} {'P1 (lower cap)':>16} {'P99 (upper cap)':>16}")
print("  " + "-" * 70)

# Fit caps on train
caps = {}
for col in cap_cols:
    if col not in df_train.columns:
        continue
    p1  = df_train[col].quantile(0.01)
    p99 = df_train[col].quantile(0.99)
    caps[col] = (p1, p99)
    print(f"  {col:<35} {p1:>16.3f} {p99:>16.3f}")

# Apply caps to all splits
def apply_caps(df, caps):
    df = df.copy()
    for col, (lower, upper) in caps.items():
        if col in df.columns:
            df[col] = df[col].clip(lower=lower, upper=upper)
    return df

df_train = apply_caps(df_train, caps)
df_val   = apply_caps(df_val, caps)
df_test  = apply_caps(df_test, caps)

print(f"\n  ✓ Caps applied to train, val, and test")

STEP 7 — Cap outliers at [1st, 99th] percentile (fit on train)

  Column                                P1 (lower cap)  P99 (upper cap)
  ----------------------------------------------------------------------
  total_price                                   11.990          989.000
  total_freight                                  7.390          105.070
  avg_price                                     10.990          899.000
  product_weight_g                              75.000        18250.000
  product_length_cm                             16.000           91.080
  product_height_cm                              2.000           65.000
  product_width_cm                              11.000           62.000
  total_payment_value                           22.380         1042.510
  product_description_lenght                    87.000         3395.000
  product_photos_qty                             1.000            8.000
  product_name_lenght                           21.000           63.000

In [11]:
print("STEP 8 — Fix data types\n")

int_cols = [
    "item_count", "unique_sellers", "unique_products",
    "payment_installments", "payment_methods_used",
    "product_photos_qty",
]

for split_name, df in [("train", df_train), ("val", df_val), ("test", df_test)]:
    for col in int_cols:
        if col in df.columns:
            df[col] = df[col].astype(int)

    # Ensure zip codes are strings (leading zeros matter in Brazil)
    for col in ["customer_zip_code_prefix", "seller_zip_code_prefix"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.zfill(5)

print("  ✓ Integer columns cast")
print("  ✓ Zip code columns cast to zero-padded strings")
print(f"\n  Sample zip codes (train):")
print(f"    customer_zip: {df_train['customer_zip_code_prefix'].head(3).tolist()}")
print(f"    seller_zip  : {df_train['seller_zip_code_prefix'].head(3).tolist()}")

STEP 8 — Fix data types

  ✓ Integer columns cast
  ✓ Zip code columns cast to zero-padded strings

  Sample zip codes (train):
    customer_zip: ['06020', '40484', '19400']
    seller_zip  : ['37564', '26379', '14940']


In [12]:
print("STEP 9 — Standardize string columns (lowercase, strip whitespace)\n")

str_cols = [
    "customer_state", "seller_state", "customer_city", "seller_city",
    "product_category_name_english", "primary_payment_type",
]

for split_name, df in [("train", df_train), ("val", df_val), ("test", df_test)]:
    for col in str_cols:
        if col in df.columns:
            df[col] = df[col].str.lower().str.strip()

print("  ✓ String columns standardized")
print(f"\n  Sample values after standardization (train):")
for col in ["customer_state", "seller_state", "primary_payment_type"]:
    print(f"    {col}: {df_train[col].unique()[:5].tolist()}")

STEP 9 — Standardize string columns (lowercase, strip whitespace)

  ✓ String columns standardized

  Sample values after standardization (train):
    customer_state: ['sp', 'ba', 'rs', 'rj', 'pa']
    seller_state: ['mg', 'rj', 'sp', 'pr', 'sc']
    primary_payment_type: ['credit_card', 'boleto', 'debit_card', 'voucher']


In [13]:
print("STEP 10 — Check and drop duplicate order_ids\n")

for split_name, df in [("train", df_train), ("val", df_val), ("test", df_test)]:
    dups = df["order_id"].duplicated().sum()
    print(f"  {split_name:<6}: duplicate order_ids = {dups}")

STEP 10 — Check and drop duplicate order_ids

  train : duplicate order_ids = 0
  val   : duplicate order_ids = 0
  test  : duplicate order_ids = 0


In [14]:
print("POST-CLEANING VALIDATION\n")
print(f"  {'Split':<8} {'Rows':>8} {'Cols':>6} {'Nulls (feature cols)':>22}")
print("  " + "-" * 50)

feature_cols_check = [
    "item_count", "total_price", "total_freight", "avg_price",
    "unique_sellers", "unique_products", "product_name_lenght",
    "product_description_lenght", "product_photos_qty",
    "product_weight_g", "product_length_cm", "product_height_cm",
    "product_width_cm", "product_category_name_english",
    "total_payment_value", "payment_installments",
    "payment_methods_used", "primary_payment_type",
    "customer_state", "seller_state",
]

for split_name, df in [("train", df_train), ("val", df_val), ("test", df_test)]:
    nulls = df[feature_cols_check].isnull().sum().sum()
    print(f"  {split_name:<8} {len(df):>8} {df.shape[1]:>6} {nulls:>22}")

print(f"\n  Target stats post-cleaning (train):")
print(df_train["lead_time_variance"].describe().round(3).to_string())

POST-CLEANING VALIDATION

  Split        Rows   Cols   Nulls (feature cols)
  --------------------------------------------------
  train       67493     40                      0
  val         14473     40                      0
  test        14471     40                      0

  Target stats post-cleaning (train):
count    67493.000
mean       -11.236
std          9.656
min        -83.071
25%        -16.245
50%        -11.989
75%         -6.401
max         97.972


In [15]:
os.makedirs("../data/processed/cleaned", exist_ok=True)

df_train.to_parquet("../data/processed/cleaned/train_clean.parquet", index=False)
df_val.to_parquet("../data/processed/cleaned/val_clean.parquet",     index=False)
df_test.to_parquet("../data/processed/cleaned/test_clean.parquet",   index=False)

# Save cleaning metadata (fit values from train — needed for deployment)
cleaning_meta = {
    "target_outlier_bounds"  : [-100, 100],
    "category_fill_value"    : CATEGORY_FILL,
    "dim_global_medians"     : global_medians,
    "text_global_medians"    : global_text_medians,
    "review_score_median"    : review_score_median,
    "outlier_caps"           : {k: list(v) for k, v in caps.items()},
}

with open("../data/processed/cleaned/cleaning_metadata.json", "w") as f:
    json.dump(cleaning_meta, f, indent=2)

print("✓ Cleaned splits saved:")
for fname in ["train_clean.parquet", "val_clean.parquet", "test_clean.parquet"]:
    path = f"../data/processed/cleaned/{fname}"
    size = os.path.getsize(path) / 1024 / 1024
    print(f"  {fname:<30} {size:.2f} MB")

print(f"\n✓ Cleaning metadata saved to cleaning_metadata.json")
print(f"\n  Final shapes:")
print(f"  Train : {df_train.shape}")
print(f"  Val   : {df_val.shape}")
print(f"  Test  : {df_test.shape}")

✓ Cleaned splits saved:
  train_clean.parquet            12.37 MB
  val_clean.parquet              2.94 MB
  test_clean.parquet             2.92 MB

✓ Cleaning metadata saved to cleaning_metadata.json

  Final shapes:
  Train : (67493, 40)
  Val   : (14473, 40)
  Test  : (14471, 40)
